In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

zones_path = "Files/nyc_taxi/lookup/taxi_zone_lookup.csv"
fx_path = "Tables/dbo/ecb_fx_raw"
gdp_path = "Tables/dbo/worldwide_bank"
air_path = "Tables/dbo/openaq"
archive_path = "Tables/dbo/openaq_archive"
silver_path = "abfss://440b1492-88fd-44d5-bac9-a81607488e15@onelake.dfs.fabric.microsoft.com/5d1d0a76-8429-4af1-967a-32514d24592c/Tables/dbo"

start_date = "2023-01-01"
end_date = "2025-12-31"
start_year = 2023
end_year = 2025
bad_zones = [264, 265]

### Zones

In [ ]:
zones = (
    spark.read.option("header", True).csv(zones_path)
    .select(
        F.col("LocationID").cast("int").alias("location_id"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone"),
        F.col("service_zone"),
    )
    .filter(~F.col("location_id").isin(bad_zones))
    .dropDuplicates(["location_id"])
)

In [ ]:
zones.write.format("delta").mode("overwrite").save(silver_path + "/taxi_zones")

### ECB FX

In [ ]:
fx_raw = (
    spark.read.format("delta").load(fx_path)
    .filter(F.col("OBS_VALUE") != "")
    .select(
        F.to_date("TIME_PERIOD").alias("date"),
        F.col("OBS_VALUE").cast("double").alias("usd_per_eur"),
    )
    .dropDuplicates(["date"])
)

In [ ]:
days = (
    fx_raw
    .agg(F.min("date").alias("first_day"), F.max("date").alias("last_day"))
    .select(F.explode(F.sequence("first_day", "last_day")).alias("date"))
)

In [ ]:
w = Window.orderBy("date")

fx = (
    days
    .join(fx_raw, "date", "left")
    .withColumn("usd_per_eur", F.last("usd_per_eur", ignorenulls=True).over(w))
    .filter(F.col("date") >= start_date)
    .filter(F.col("date") <= end_date)
)

In [ ]:
fx.write.format("delta").mode("overwrite").save(silver_path + "/ecb_fx")

### GDP

In [ ]:
gdp = (
    spark.read.format("delta").load(gdp_path)
    .select(
        F.col("countryiso3code").alias("country_code"),
        F.col("date").cast("int").alias("year"),
        F.col("value").alias("gdp_usd"),
    )
    .filter(F.col("gdp_usd").isNotNull())
    .filter(F.col("year") >= start_year)
    .filter(F.col("year") <= end_year)
    .dropDuplicates(["country_code", "year"])
)

In [ ]:
gdp.write.format("delta").mode("overwrite").save(silver_path + "/gdp")

### OpenAQ

In [ ]:
air_api = (
    spark.read.format("delta").load(air_path)
    .withColumn("datetime_utc", F.to_timestamp("datetime_utc"))
)

stations = air_api.select("location_id", "location_name").distinct()

In [ ]:
air_archive = (
    spark.read.format("delta").load(archive_path)
    .filter(F.col("parameter").isin("pm25", "no2"))
    .select(
        "location_id",
        F.col("sensors_id").alias("sensor_id"),
        "parameter",
        "units",
        F.to_timestamp("datetime").alias("datetime_utc"),
        "value",
    )
    .join(stations, "location_id")
    .join(air_api, ["location_id", "parameter", "datetime_utc"], "left_anti")
)

In [ ]:
air = (
    air_api.unionByName(air_archive)
    .filter(F.col("value").isNotNull())
    .filter(F.col("value") >= 0)
    .withColumn("datetime_local", F.from_utc_timestamp("datetime_utc", "America/New_York"))
    .withColumn("date", F.to_date("datetime_local"))
    .withColumn("hour", F.hour("datetime_local"))
    .filter(F.col("date") >= start_date)
    .filter(F.col("date") <= end_date)
    .dropDuplicates(["sensor_id", "datetime_utc"])
    .select(
        F.col("location_id").cast("int"),
        "location_name",
        F.col("sensor_id").cast("int"),
        "parameter",
        "units",
        "datetime_utc",
        "datetime_local",
        "date",
        "hour",
        "value",
    )
)

In [ ]:
air.write.format("delta").mode("overwrite").save(silver_path + "/air_quality")

### Check

In [ ]:
for name in ["taxi_zones", "ecb_fx", "gdp", "air_quality"]:
    print(name, spark.read.format("delta").load(silver_path + "/" + name).count())